# Part 3: Classification with Gaussian Mixture Data

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

**(a)** Plot the data points in Dtrain ∪ Dtest using different colors to indicate classes and different symbols to indicate train vs. test set. **(2 points)**


## Question (a): Generate the dataset

In [ ]:
np.random.seed(42)

def generate_dataset(N, eps, p):
    """Generate a Gaussian mixture dataset.

    Class 1 (probability p):  X ~ N(mu_0, Sigma_0) where mu_0=[0,0], Sigma_0=0.5*I
    Class 0 (probability 1-p): X ~ N(mu_1, Sigma_1) where mu_1=[eps,0], Sigma_1=0.4*I
    """
    mu_0 = np.array([0.0, 0.0])
    sg_0 = np.array([[0.5, 0.0], [0.0, 0.5]])
    mu_1 = np.array([eps, 0.0])
    sg_1 = np.array([[0.4, 0.0], [0.0, 0.4]])

    Y = np.zeros(N, dtype=int)
    X = np.zeros((N, 2))

    for i in range(N):
        Bi = np.random.uniform()
        if Bi < p:
            Y[i] = 1
            X[i] = np.random.multivariate_normal(mu_0, sg_0)
        else:
            Y[i] = 0
            X[i] = np.random.multivariate_normal(mu_1, sg_1)

    return X, Y

In [ ]:
N_train = 50
N_test  = 1000
eps     = 1
p       = 0.2

X_train, y_train = generate_dataset(N=N_train, eps=eps, p=p)
X_test,  y_test  = generate_dataset(N=N_test,  eps=eps, p=p)

print(f"Training set: {X_train.shape}, class balance: {y_train.mean():.2f}")
print(f"Test set:     {X_test.shape},  class balance: {y_test.mean():.2f}")

## Question (a): Scatter plot

Train points are plotted as circles (`o`), test points as triangles (`^`). Class 0 is blue, class 1 is red.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

# Training points
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
           c='blue',  marker='o', label='Train class 0')
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
           c='red',   marker='o', label='Train class 1')

# Test points
ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1],
           c='blue',  marker='^', alpha=0.4, label='Test class 0')
ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1],
           c='red',   marker='^', alpha=0.4, label='Test class 1')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('X_1')
ax.set_ylabel('X_2')
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Gaussian Mixture Dataset')
plt.tight_layout()
plt.show()

**(b)** What is the mathematical expression for the optimal Bayes classifier in this setting? And for its boundary region? Do you expect it to be linear? **(1 point)**


**(c)** Estimate the error of the Bayes classifier on the samples from Dtest. How do you expect it to change in terms of ε? **(1 point)**


## Question (b) & (c): Bayes Classifier

The posterior ratio is:
$$
R(x_1, x_2) = \exp\!\Big(0.25\,x_1^2 - 2.5\,x_1\varepsilon + 1.25\,\varepsilon^2\Big) \times \frac{0.8\,p}{1-p}
$$
and the classifier predicts class 1 if $R > 1$.

In [ ]:
def bayes_classifier(x, eps, p):
    """Bayes-optimal classifier for the Gaussian mixture model."""
    x1 = x[0]
    R = np.exp(0.25 * x1**2 - 2.5 * x1 * eps + 1.25 * eps**2) * 0.8 * p / (1 - p)
    return 1 if R > 1 else 0

y_pred_bayes = np.array([bayes_classifier(X_test[i], eps, p) for i in range(N_test)])
acc_bayes = np.mean(y_pred_bayes == y_test)
print(f"Score with Bayes classifier: {acc_bayes:.2f}")

**(d)** Given the structure of the model generating the datasets, which classifier (from our lectures) would you expect to be the most adequate? **(0.5 points)**


**(e)** Train a LDA, a QDA, and a Logistic Regression classifier on Dtrain and estimate their errors on Dtest. How do their errors compare to the value obtained in (b)? **(0.5 points)**

---


## Question (d) & (e): LDA, QDA, and Logistic Regression

In [ ]:
# LDA
clf_lda = LinearDiscriminantAnalysis()
clf_lda.fit(X_train, y_train)
acc_lda = clf_lda.score(X_test, y_test)
print(f"Score with LDA: {acc_lda:.2f}")

In [ ]:
# QDA
clf_qda = QuadraticDiscriminantAnalysis()
clf_qda.fit(X_train, y_train)
acc_qda = clf_qda.score(X_test, y_test)
print(f"Score with QDA: {acc_qda:.2f}")

In [ ]:
# Logistic Regression
clf_lgr = LogisticRegression(max_iter=1000)
clf_lgr.fit(X_train, y_train)
acc_lgr = clf_lgr.score(X_test, y_test)
print(f"Score with Logistic Regression: {acc_lgr:.2f}")

## Question (f): Transfer Learning — test with p = 0.8

We now generate a new test set where the class prior has changed from p=0.2 to p=0.8. The classifiers were trained on data with p=0.2, so we are evaluating their ability to transfer to a different distribution.

In [ ]:
p_new = 0.8
X_test_prime, y_test_prime = generate_dataset(N=N_test, eps=eps, p=p_new)

acc_lda_tl = clf_lda.score(X_test_prime, y_test_prime)
acc_qda_tl = clf_qda.score(X_test_prime, y_test_prime)
acc_lgr_tl = clf_lgr.score(X_test_prime, y_test_prime)

print(f"Transfer learning (p={p_new}):")
print(f"  Score with LDA:                 {acc_lda_tl:.2f}")
print(f"  Score with QDA:                 {acc_qda_tl:.2f}")
print(f"  Score with Logistic Regression: {acc_lgr_tl:.2f}")

## Discussion

**Original test set (p=0.2):** The Bayes classifier achieves the highest accuracy, as expected — it is the theoretically optimal rule for this data distribution. LDA, QDA, and logistic regression all perform slightly below the Bayes classifier because they must estimate parameters from a finite training set.

**Transfer learning (p=0.8):** All three classifiers show a drop in performance. This is not surprising: the classifiers were trained on data where class 1 was rare (p=0.2), but the new test distribution has a much higher prevalence of class 1 (p=0.8). The learned decision boundaries and implicit prior assumptions no longer match the test distribution — a classic transfer learning challenge.

The parameter $\varepsilon$ controls the separation between the two class-conditional distributions. Larger $\varepsilon$ makes the problem easier and all classifiers achieve higher accuracy.